# **How Economic Stress Shapes Musical Taste in the United States**

---



Students: Hoang Dieu Linh Nguyen, Tri Huynh

Project Pages:

[Project Website](https://evelynnguyenh.github.io)

[Project Repository Page](https://github.com/evelynnguyenh/evelynnguyenh.github.io)

### **About This Project**

Music reflects the social and cultural mood of a nation. During periods of economic hardship, people's spending, employment, and general outlook shift dramatically. We believe this stress may be reflected in the type of music that dominates the charts. This project investigates whether economic recessions in the United States are associated with a measurable shift in the emotional tone of popular music, and whether Americans gravitate toward more melancholic or more escapist music during difficult economic periods.





### **Project Datasets**
The datasets we are considering for the project are:

**Dataset 1:** Billboard Hot 100
Source: https://www.kaggle.com/datasets/dhruvildave/billboard-the-hot-100-songs

**Dataset 2:** Spotify Tracks 1921-2020
Source: https://www.kaggle.com/datasets/yamaerenay/spotify-dataset-19212020-600k-tracks

**Dataset 3:** GDP-Based Recession Indicator (FRED)
Source: https://fred.stlouisfed.org/series/JHDUSRGDPBR


The Billboard Hot 100, published weekly since 1958, records which songs were most popular in the United States at any given time. The Spotify Tracks Dataset provides audio features including valence, energy, and danceability for over 600,000 real tracks sourced from Spotify's API. The FRED recession indicator marks which quarters were classified as recessions based on GDP data, serving as our economic anchor.



### **Collaboration Plan**
The team plans to meet in-person twice a week to review progress and align on next steps. Our primary communication channel is text messaging for quick updates between meetings. Work is divided equally and tracked through a shared Google Sheets task list with assigned to-do items. All project files are stored in a shared GitHub repository, which also hosts our GitHub Pages site, and a shared Google Drive folder for any additional resources. The notebook is maintained in Google Colab, with us coordinating over text before editing to avoid conflicts.

### **Project Goals**
By combining the three datasets above, we aim to determine whether economic recessions in the United States are associated with measurable shifts in the musical characteristics of popular songs. Specifically, we want to find out whether Americans gravitate toward sadder, lower-energy music during periods of economic hardship, which audio features shift most, and whether any effect is immediate or lagged.

### **ETL (Extraction, Transform, and Load)**

First, we will load in the libraries we'll be using for this tutorial.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#### Loading the Datasets

We load all three datasets directly from our GitHub repository using their raw URLs, making the notebook reproducible for anyone without needing local file access.

In [ ]:
# Load the datasets
songs = pd.read_csv('https://media.githubusercontent.com/media/evelynnguyenh/evelynnguyenh.github.io/refs/heads/main/songs.csv')
fed_recession = pd.read_csv('https://raw.githubusercontent.com/evelynnguyenh/evelynnguyenh.github.io/refs/heads/main/recession_data.csv')
billboard = pd.read_csv('https://raw.githubusercontent.com/evelynnguyenh/evelynnguyenh.github.io/refs/heads/main/charts.csv')

#### [Billboard Top 100](https://www.kaggle.com/datasets/dhruvildave/billboard-the-hot-100-songs)
The Hot 100 is compiled weekly by Billboard magazine using a combination of
digital song sales, radio airplay, and streaming data across the United States. Through this dataset, we can answer the question:

**Do songs stay on the Billboard Hot 100 longer during recession periods compared to periods of economic growth**?


Let's look at this dataset and examine its structure.

In [ ]:
print("Shape:", billboard.shape, "\n")
print("Column names and dtypes:", billboard.dtypes, "\n")
print("Null counts per column:")
print(billboard.isnull().sum())
billboard.head(10)

We can make some observations about this dataset:
- Each row represents a single song's appearance on the Billboard Hot 100 for a specific week, giving us 330,087 weekly chart entries in total.
- The `date` column is stored as a string object rather than a datetime64 dtype, which we will need to convert to the correct dtype.
- The `last-week` column contains 32,312 NaN values. This indicates that those songs were not on the chart the previous week, meaning it is a new entry.
- As a result of those NaNs, `last-week` is stored as float64 (e.g. 1.0, 2.0) instead of an integer.
- Column names use hyphens (`last-week`, `peak-rank`, `weeks-on-board`), which can cause errors so we will rename them to use underscores.
- The dataset is already tidy with one row per observation, one column per variable, so no restructuring is needed.

Convert the dtype of `date`

In [ ]:
# Convert date from string to datetime
billboard['date'] = pd.to_datetime(billboard['date'])

# Verify that it worked
print(billboard['date'].dtype)
billboard['date'].head()

Now let's rename columns to use an underscore instead of hyphen. Then, we convert the dtype from float64 to Int64 since rank values should be whole numbers, and Int64 also supports NaN values. We want to keep NaN here because it's a meaningful value which indicates a song's first appearance on the chart.

In [ ]:
# Rename hyphen to underscore
billboard = billboard.rename(columns={
    'last-week': 'last_week',
    'peak-rank': 'peak_rank',
    'weeks-on-board': 'weeks_on_board'
})

# Convert last_week from float64 to nullable integer
# Int64 (capital I) supports NaN, regular int64 does not
billboard['last_week'] = billboard['last_week'].astype('Int64')

# Verify
print(billboard.dtypes)
print()
print("NaN count in last_week:", billboard['last_week'].isna().sum())

#### [Spotify Tracks Dataset 1921-2020](https://www.kaggle.com/datasets/yamaerenay/spotify-dataset-19212020-600k-tracks)

This dataset contains audio features for over 600,000 tracks sourced directly from Spotify's API, spanning nearly a century of recorded music. Each track includes attributes such as valence, energy, danceability, and popularity score. We will use this dataset to measure the emotional tone of popular songs over time.

This dataset can help us answer questions, such as:

 **Does the average valence of songs decrease during recession periods, suggesting a shift toward more negative or somber music?**

In [ ]:
# Preview Spotify dataset
print("Shape:", songs.shape)
print()
print(songs.dtypes)
songs.head()

#### [GDP-Based Recession Indicator (FRED)](https://fred.stlouisfed.org/series/JHDUSRGDPBR)

This dataset from the Federal Reserve Bank of St. Louis marks whether the U.S. economy was in a recession during each quarter, based solely on GDP data. It serves as our economic anchor for comparing against music trends. This dataset is central to answering:

**How the audio features of Billboard Hot 100 songs during recession periods compare to those during periods of economic growth?**

In [ ]:
# Preview FRED recession indicator
print("Shape:", fed_recession.shape)
print()
print(fed_recession.dtypes)
fed_recession.head()

## **EDA (Exploratory Data Analysis)**

#### [Billboard Top 100](https://www.kaggle.com/datasets/dhruvildave/billboard-the-hot-100-songs)

1. Let's get a more general overview of our dataset to get a general overview of song popularity duration, we examine summary statistics for the weeks_on_board variable.

In [ ]:
billboard['weeks_on_board'].describe()

From the summary statistics, we observe that the median number of weeks a song stays on the chart is 7 weeks, meaning that half of chart entries last longer than seven weeks. The average duration is about 9.16 weeks, suggesting that while most songs remain on the chart for a relatively short period, some songs stay much longer and increase the overall mean. The maximum value of 90 weeks indicates that a small number of songs achieve exceptional long-term popularity.

2. Examining the artists who appear most frequently in the Billboard dataset helps identify musicians who consistently produce charting songs and dominate popular music trends over time.

In [ ]:
billboard['artist'].value_counts().head(10)

We plot the distribution of weeks that songs stay on the Billboard Hot 100 chart to understand how long songs typically remain popular. This provides a baseline measure of song popularity that can later be compared across different economic periods.

3. We also examine how many songs reached the number one position on the Billboard Hot 100 chart to understand the frequency of chart-topping hits in the dataset

In [ ]:
(billboard['peak_rank'] == 1).sum()

We also examine how many songs reached the number one position on the Billboard Hot 100 chart to understand the frequency of chart-topping hits in the dataset. The result shows 16,827 entries with a peak rank of 1, representing major chart-topping hits in the dataset.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(billboard['weeks_on_board'], bins=40)
plt.title("Distribution of Weeks on Billboard Chart")
plt.xlabel("Weeks on Chart")
plt.ylabel("Count")
plt.show()

This distribution shows that most songs remain on the chart for fewer than 15 weeks, while only a small number of songs stay for much longer periods. Understanding how long songs remain on the chart gives us a baseline for song popularity. In later stages of the project, we can examine whether song popularity duration changes during economic recessions.

### **Mount Google Drive and Export**

The cells below are used to convert this notebook to HTML for our project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%shell
jupyter nbconvert --to html '/content/drive/MyDrive/CMPS3160 Project/Data Science Project.ipynb'